In [ ]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Configuration
INPUT_DIR = Path("..") / "data" / "raw"
OUTPUT_DIR = Path("..") / "data" / "pre-procest"
TARGET_SIZE = (240, 240)  # (Width, Height)
# Replace these with the output from the calibration script
K = np.array([[ 342.12, 0., 120.45], [0., 341.98, 121.12], [0., 0., 1.]]) # Example
D = np.array([[-0.05], [0.02], [-0.01], [0.001]]) # Example

def preprocess_image(img: np.ndarray) -> np.ndarray:
    """
    Standardized pipeline for both Python training and C++ inference.
    """
    # 1. Undistort (Fisheye specific)
    # We use initUndistortRectifyMap to make it faster for batch processing
    h, w = img.shape[:2]
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), K, (w, h), cv2.CV_16SC2)
    img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR)

    # 2. Rotate 90 degrees counter-clockwise (Bebop 1 sensor orientation)
    # C++: cv::rotate(img, img, cv::ROTATE_90_COUNTERCLOCKWISE);
    img = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)

    # 3. Resize using Linear Interpolation (Faster on ARM than INTER_AREA)
    # C++: cv::resize(img, img, cv::Size(240, 240), 0, 0, cv::INTER_LINEAR);
    img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_LINEAR)

    # 4. Gaussian Blur (Fast smoothing to handle sensor noise)
    # C++: cv::GaussianBlur(img, img, cv::Size(3, 3), 0);
    img = cv2.GaussianBlur(img, (3, 3), 0)
    # 5. Normalize
    # C++: img.convertTo(img, CV_32F, 1.0 / 255.0);  (this is necessary for runtime pre_processing)
    img = img.astype(np.float32) / 255.0

    return img

def process_dataset():
    # Setup directories
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    files = [p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in extensions]

    print(f"Processing {len(files)} images...")

    for src_path in tqdm(files):
        img = cv2.imread(str(src_path))
        if img is None: continue

        # Run the C-compatible pipeline
        processed = preprocess_image(img)
        processed = (processed * 255).astype(np.uint8)  # Convert back to uint8 for saving
        # Save the result
        # Note: We save as uint8 (0-255) so you can use LabelImg for annotations.
        dst_path = OUTPUT_DIR / src_path.name
        cv2.imwrite(str(dst_path), processed)

if __name__ == "__main__":
    process_dataset()
    print("Pre-processing complete. Images ready for LabelImg.")